In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/gowrishankarp/newspaper-text-summarization-cnn-dailymail/cnn_dailymail/validation.csv
/kaggle/input/datasets/gowrishankarp/newspaper-text-summarization-cnn-dailymail/cnn_dailymail/train.csv
/kaggle/input/datasets/gowrishankarp/newspaper-text-summarization-cnn-dailymail/cnn_dailymail/test.csv


In [2]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.cuda.amp import GradScaler, autocast
from torch.utils.checkpoint import checkpoint
from transformers import GPT2Tokenizer
import pandas as pd
import re
import html
import unicodedata
import math
import os

In [4]:
class AdvancedDataPreprocessor:
    """
    Advanced Cleaning and Normalization Protocol specifically engineered 
    for the Kaggle CNN/DailyMail News Summarization dataset.
    """
    
    # 1. Standard Web & Unicode Artifacts
    HTML_TAGS_RE = re.compile(r'<.*?>')
    URL_RE = re.compile(r'https?://\S+|www\.\S+')
    EMAIL_RE = re.compile(r'\S+@\S+')
    TWITTER_HANDLE_RE = re.compile(r'@\w+')
    NON_ASCII_RE = re.compile(r'[^\x00-\x7F]+')
    MULTI_WS_RE = re.compile(r'\s+')
    
    # 2. DailyMail Specific Metadata Block (Highly robust)
    # Catches combinations of "By...", "| . UPDATED...", and "PUBLISHED..."
    DAILYMAIL_META_RE = re.compile(
        r'(?i)^'                                     # Start of string, case-insensitive
        r'(?:By\b.{0,100}?\.?\s*)?'                  # Optional 'By' line (up to 100 chars)
        r'(?:\|\s*\.?\s*)?'                          # Optional leading pipe e.g., "| ." or "|"
        r'(?:PUBLISHED|UPDATED)\b.{0,100}?(?:19|20)\d{2}\s*\.?\s*' # 1st date block + Year
        r'(?:\|\s*\.?\s*(?:PUBLISHED|UPDATED)\b.{0,100}?(?:19|20)\d{2}\s*\.?\s*)?' # Optional 2nd date block
    )
    
    # 3. Standard Agency Datelines & Standalone Bylines
    DATELINE_RE = re.compile(r'^[A-Z\s]+(?:[,-]\s*[A-Z\s]+)*\s*(?:\([^)]+\))?\s*(?:--|-|—)\s*')
    BYLINE_RE = re.compile(r'(?i)^By\s+[A-Z\s,]+(?:[.-]\s*)?')
    
    # 4. Mid-text Journalistic Boilerplate
    BOILERPLATE_RE = re.compile(
        r'(?i)(?:scroll down for video|click here to read more|read more:'
        r'|watch the video|share this article|related articles).*'
    )

    @classmethod
    def standardize_text(cls, text: str) -> str:
        if pd.isna(text) or not isinstance(text, str):
            return ""
            
        text = html.unescape(text)
        text = cls.HTML_TAGS_RE.sub('', text)
        text = cls.URL_RE.sub('', text)
        text = cls.EMAIL_RE.sub('', text)
        text = cls.TWITTER_HANDLE_RE.sub('', text)
        text = unicodedata.normalize('NFKD', text)
        
        replacements = {
            '“': '"', '”': '"', '‘': "'", '’': "'",
            '—': '-', '–': '-', '…': '...', '\n': ' ', '\t': ' '
        }
        for search, replace in replacements.items():
            text = text.replace(search, replace)
            
        # Strict ASCII enforcer (drops Emojis)
        text = cls.NON_ASCII_RE.sub('', text)
        return text

    @classmethod
    def remove_artifacts(cls, text: str) -> str:
        # 1. Remove standard Datelines ("WASHINGTON (Reuters) -- ")
        text = cls.DATELINE_RE.sub('', text)
        
        # 2. TARGETED: Remove DailyMail Metadata Blocks
        text = cls.DAILYMAIL_META_RE.sub('', text)
        
        # 3. Remove standalone Bylines
        text = cls.BYLINE_RE.sub('', text)
        
        # 4. Remove mid-text boilerplate
        text = cls.BOILERPLATE_RE.sub('', text)
        
        # 5. Clean up messy whitespace
        text = cls.MULTI_WS_RE.sub(' ', text).strip()
        
        return text

    @classmethod
    def process_dataset(cls, df: pd.DataFrame, sample_size: int = None, random_state: int = 42) -> pd.DataFrame:
        df = df.dropna(subset=['article', 'highlights']).copy()
        df = df.drop_duplicates(subset=['article']).copy()
        
        if sample_size is not None and len(df) > (sample_size * 2):
            df = df.sample(n=sample_size * 2, random_state=random_state)
            
        df['article'] = [cls.remove_artifacts(cls.standardize_text(text)) for text in df['article']]
        df['highlights'] = [cls.standardize_text(text).strip() for text in df['highlights']]
        
        def is_valid_pair(row):
            art_words = row['article'].split()
            sum_words = row['highlights'].split()
            if len(art_words) < 50 or len(sum_words) < 5:
                return False
            ratio = len(sum_words) / len(art_words)
            return (0.05 <= ratio <= 0.60)
            
        mask = df.apply(is_valid_pair, axis=1)
        df_filtered = df[mask].reset_index(drop=True)
        
        if sample_size is not None:
            actual_sample = min(sample_size, len(df_filtered))
            df_filtered = df_filtered.sample(n=actual_sample, random_state=random_state).reset_index(drop=True)
            
        return df_filtered

In [5]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model=768, n_heads=12):
        super().__init__()
        self.ln_1 = nn.LayerNorm(d_model)
        # We use batch_first=True for efficiency
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ln_2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model)
        )

    def forward(self, x, mask):
        # We pass the mask as an argument to the block to avoid 
        # slicing issues inside the checkpoint function
        def _block(x, m):
            ln1_x = self.ln_1(x)
            attn_out, _ = self.attn(ln1_x, ln1_x, ln1_x, attn_mask=m)
            x = x + attn_out
            x = x + self.mlp(self.ln_2(x))
            return x
        
        return checkpoint(_block, x, mask, use_reentrant=False)

class AbstractiveGPT2(nn.Module):
    def __init__(self, vocab_size=50257, d_model=768, n_layers=6):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.wte = nn.Embedding(vocab_size, d_model)
        self.wpe = nn.Embedding(1024, d_model)
        self.blocks = nn.ModuleList([TransformerBlock(d_model) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.xavier_uniform_(m.weight) # More stable than normal_
            if isinstance(m, nn.Linear) and m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, idx):
        # 1. Device-safe mask generation (Avoids index-out-of-bounds asserts)
        T = idx.size(1)
        device = idx.device
        mask = torch.triu(torch.ones(T, T, device=device), diagonal=1).bool()
        
        # 2. Bound check
        idx = torch.clamp(idx, max=self.vocab_size - 1)
            
        pos = torch.arange(T, device=device)
        x = self.wte(idx) + self.wpe(pos)
        
        for block in self.blocks: 
            x = block(x, mask)
            
        return self.lm_head(self.ln_f(x))

    def resize_token_embeddings(self, new_size):
        # Re-initialize to the correct size completely to avoid assertion errors
        self.vocab_size = new_size
        old_weight = self.wte.weight.data
        self.wte = nn.Embedding(new_size, self.d_model).to(old_weight.device)
        self.wte.weight.data.normal_(std=0.02)
        n = min(new_size, old_weight.size(0))
        self.wte.weight.data[:n, :] = old_weight[:n, :]
        self.lm_head = nn.Linear(self.d_model, new_size, bias=False)
        self.lm_head.weight = self.wte.weight

In [6]:
def setup_tokenizer():
    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    tokenizer.add_special_tokens({'additional_special_tokens': ['<|startoftext|>', '<|endoftext|>'], 'pad_token': '<|endoftext|>'})
    return tokenizer

class CNNDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=1024):
        self.data = df
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.sep = "\n\nTL;DR:\n"

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        # Construct strings
        prompt = f"<|startoftext|>{row['article']}{self.sep}"
        full_text = f"{prompt}{row['highlights']}<|endoftext|>"
        
        # Tokenize with strict truncation
        enc = self.tokenizer(full_text, truncation=True, max_length=self.max_len)
        ids = torch.tensor(enc.input_ids, dtype=torch.long)
        
        # Determine prefix length based on the actual tokens generated
        # This is safer than re-encoding the string
        prompt_ids = self.tokenizer.encode(prompt, add_special_tokens=False)
        # Handle case where truncation might have cut into the prompt
        prefix_len = min(len(ids), len(prompt_ids) + 1) # +1 for start token
        
        labels = ids.clone()
        labels[:prefix_len] = -100
        return ids, labels

def collate_fn(batch):
    ids, labels = zip(*batch)
    return (torch.nn.utils.rnn.pad_sequence(ids, batch_first=True, padding_value=50256),
            torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=-100))

from torch.cuda.amp import GradScaler, autocast
import torch.nn.functional as F

class MemoryEfficientTrainer:
    def __init__(self, model, optimizer, device, grad_accum_steps=8, max_grad_norm=1.0):
        self.model = model
        self.optimizer = optimizer
        self.device = device
        self.grad_accum_steps = grad_accum_steps
        self.max_grad_norm = max_grad_norm
        self.scaler = GradScaler()

    def train(self, dataloader, epochs):
        self.model.train()
        for epoch in range(epochs):
            for i, (ids, labels) in enumerate(dataloader):
                ids, labels = ids.to(self.device), labels.to(self.device)
                
                with torch.amp.autocast('cuda'):
                    logits = self.model(ids)
                    # Shift for next-token prediction
                    shift_logits = logits[:, :-1, :].contiguous()
                    shift_labels = labels[:, 1:].contiguous()
                    
                    loss = F.cross_entropy(
                        shift_logits.view(-1, shift_logits.size(-1)), 
                        shift_labels.view(-1), 
                        ignore_index=-100
                    ) / self.grad_accum_steps
                
                # Scaled backward pass
                self.scaler.scale(loss).backward()
                
                if (i + 1) % self.grad_accum_steps == 0:
                    # 1. Unscale gradients for clipping
                    self.scaler.unscale_(self.optimizer)
                    
                    # 2. Gradient Clipping (Prevents loss spikes)
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.max_grad_norm)
                    
                    # 3. Step and Update
                    self.scaler.step(self.optimizer)
                    self.scaler.update()
                    self.optimizer.zero_grad()
                    
                    # 4. Explicit Memory Cleanup
                    torch.cuda.empty_cache()
                    
                    print(f"Epoch {epoch} | Step {i} | Loss: {loss.item()*self.grad_accum_steps:.4f}")

In [7]:
@torch.no_grad()
def generate_summary(model, tokenizer, article, max_gen_length=150, temperature=0.7, top_p=0.92, device='cuda'):
    model.eval()
    
    # 1. Prepare prefix as a tensor immediately
    prefix = f"<|startoftext|>{article}\n\nTL;DR:\n"
    S = tokenizer.encode(prefix, return_tensors='pt').to(device)
    end_token = tokenizer.convert_tokens_to_ids("<|endoftext|>")
    
    # 2. Use a loop that minimizes memory allocation
    for _ in range(max_gen_length):
        if S.size(1) >= 1024: break
        
        # Forward pass on last token only if possible (for efficiency)
        logits = model(S)[:, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)
        
        # Nucleus sampling
        sorted_probs, sorted_indices = torch.sort(probs, descending=True)
        cumulative = torch.cumsum(sorted_probs, dim=-1)
        
        # Find threshold
        mask = cumulative > top_p
        mask[..., 1:] = mask[..., :-1].clone()
        mask[..., 0] = False
        
        probs[mask.scatter(1, sorted_indices, mask)] = 0
        token_next = torch.multinomial(probs, 1)
        
        if token_next.item() == end_token: break
        S = torch.cat((S, token_next), dim=1)
        
    # Cleanup memory
    del logits, probs
    torch.cuda.empty_cache()
    
    return tokenizer.decode(S[0][len(tokenizer.encode(prefix)):], skip_special_tokens=True)

In [8]:
def run_full_pipeline(raw_df):
    print("--- 1. LOADING & PREPROCESSING ---")
    df_clean = AdvancedDataPreprocessor.process_dataset(raw_df, sample_size=2000)
    
    print("--- 2. INITIALIZING MODEL ---")
    tokenizer = setup_tokenizer()
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    
    # Kaggle memory safe configuration
    d_model = 384 
    n_layers = 6
    model = AbstractiveGPT2(vocab_size=50257, d_model=d_model, n_layers=n_layers).cpu()
    
    model.resize_token_embeddings(len(tokenizer))
    model = model.to('cuda')
    
    print("--- 3. SETTING UP TRAINING ---")
    dataset = CNNDataset(df_clean, tokenizer)
    # Changed batch_size to 1 and added gradient accumulation logic
    dataloader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    # Using grad_accum_steps=32 to simulate batch_size=32 while keeping VRAM usage for batch_size=1
    trainer = MemoryEfficientTrainer(model, optimizer, device='cuda', grad_accum_steps=32)
    
    print("--- 4. STARTING TRAINING ---")
    trainer.train(dataloader, epochs=8)
    
    # --- SAVING THE MODEL ---
    print("\n--- 5. SAVING MODEL ---")
    save_path = "/kaggle/working/gpt2_cnn_summarizer.pt"
    torch.save(model.state_dict(), save_path)
    print(f"Model saved to {save_path}")
    
    # Save the tokenizer as well
    tokenizer.save_pretrained("/kaggle/working/tokenizer")
    print("Tokenizer saved.")
    
    print("--- 6. INFERENCE ---")
    # Set to eval mode before inference
    model.eval()
    test_text = """
        Can you predict solar radiation when sensors can't be trusted?
        The Trans-African Hydro-Meteorological Observatory (TAHMO) operates over 600 weather stations across 
        twenty-three African countries, providing the largest in-situ weather dataset in Africa. Maintaining high-quality 
        observations across such a distributed network is essential for climate research, agriculture, water management, and 
        renewable energy planning. Measuring incoming shortwave radiation reliably over long periods is challenging. 
        The sensors are factory-calibrated at installation. However, after approximately two years of exposure to environmental conditions, 
        sensors begin to drift in a nonlinear manner. This drift introduces systematic bias into radiation measurements, 
        compromising data quality and downstream applications. The goal is to develop a machine learning solution capable of 
        reconstructing true incoming shortwave radiation and enabling operational drift correction across TAHMO's network.
        Your task for this challenge is to predict incoming shortwave radiation at 15-minute interval for the missing even 
        months of year 1 for each station.
    """
    generated = generate_summary(model, tokenizer, test_text, device='cuda')
    print(f"Generated Summary: {generated}")

if __name__ == '__main__':
    import os
    os.environ['CUDA_LAUNCH_BLOCKING'] = "1" 
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    
    # Load dataset
    file_path = "/kaggle/input/datasets/gowrishankarp/newspaper-text-summarization-cnn-dailymail/cnn_dailymail/train.csv"
    dirty_df = pd.read_csv(file_path)
    
    run_full_pipeline(dirty_df)

--- 1. LOADING & PREPROCESSING ---
--- 2. INITIALIZING MODEL ---


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

--- 3. SETTING UP TRAINING ---


/tmp/ipykernel_23/1514775290.py:50: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler()


--- 4. STARTING TRAINING ---


Token indices sequence length is longer than the specified maximum sequence length for this model (1318 > 1024). Running this sequence through the model will result in indexing errors


Epoch 0 | Step 31 | Loss: 10.8478
Epoch 0 | Step 63 | Loss: 10.7179
Epoch 0 | Step 95 | Loss: 10.6897
Epoch 0 | Step 127 | Loss: 10.6266
Epoch 0 | Step 159 | Loss: 10.5794
Epoch 0 | Step 191 | Loss: 10.5177
Epoch 0 | Step 223 | Loss: 10.4867
Epoch 1 | Step 31 | Loss: 10.4337
Epoch 1 | Step 63 | Loss: 10.3987
Epoch 1 | Step 95 | Loss: 10.3357
Epoch 1 | Step 127 | Loss: 10.3113
Epoch 1 | Step 159 | Loss: 10.2512
Epoch 1 | Step 191 | Loss: 10.2065
Epoch 1 | Step 223 | Loss: 10.1945
Epoch 2 | Step 31 | Loss: 10.1274
Epoch 2 | Step 63 | Loss: 10.0913
Epoch 2 | Step 95 | Loss: 10.0394
Epoch 2 | Step 127 | Loss: 10.0110
Epoch 2 | Step 159 | Loss: 9.9382
Epoch 2 | Step 191 | Loss: 9.9242
Epoch 2 | Step 223 | Loss: 9.7892
Epoch 3 | Step 31 | Loss: 9.8393
Epoch 3 | Step 63 | Loss: 9.8100
Epoch 3 | Step 95 | Loss: 9.6890
Epoch 3 | Step 127 | Loss: 9.6621
Epoch 3 | Step 159 | Loss: 9.6482
Epoch 3 | Step 191 | Loss: 9.6411
Epoch 3 | Step 223 | Loss: 9.5335
Epoch 4 | Step 31 | Loss: 9.5475
Epoch 4 |